# Module 5A — Generative AI & LLM Foundations

Welcome to a hands-on engineering workshop for Large Language Models. This notebook is designed for live teaching, with clear explanations, runnable examples, and practical code that shows how LLM APIs behave in real engineering workflows.

## Course Flow

1. What is an LLM actually doing?
2. Using LLM APIs for real-world tasks
3. Prompt structuring & iteration
4. Temperature & randomness
5. Hallucinations and safeguards
6. Evaluation metrics and engineering checks
7. ML vs LLM comparison
8. Cost, latency, reliability, and retries

This notebook is sequential, beginner-friendly, and engineered to support live explanations and coding demonstrations.

# Slides 7-10

## Required Imports and API Setup


Before we run code, we need to explain environment variables, API keys, and safe client setup.

### Environment variables and API keys

- Environment variables are configuration values stored outside your source code.
- API keys must never be hardcoded in notebooks, scripts, or repos.
- If a key is leaked, anyone can use your account and incur costs.
- Use `.env` files locally and a secrets manager in production.

### OpenAI client setup

We use `dotenv` to load environment variables and `OpenAI` from the OpenAI Python client. The client reads `OPENAI_API_KEY` from the environment, so secrets remain separate from code.

In [4]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
import time

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [5]:
os.getenv("OPENAI_API_KEY")[:10]

'sk-proj-8W'

Expected Observation:
- The imports should run without errors if `python-dotenv` and `openai` are installed.
- The `client` object is now ready to send requests to the OpenAI API.

# Part 1 — What is an LLM actually doing?

This section explains the core mechanics and how engineers should think about generation.

### Core Idea

A Large Language Model is fundamentally:

- a next-token prediction system
- trained on massive text corpora
- probabilistic, not deterministic

For example, the model sees the prefix:

"The capital of France is" → "Paris"

and predicts the most likely next token given the context.

### Statistical prediction and sampling

- The model computes a probability distribution over possible next tokens.
- It is not choosing the answer from a fixed database.
- Token sampling can introduce variability even for the same prompt.

The model may assign probabilities like this:

Token | Probability
--- | ---
morning | 0.55
office | 0.12
evening | 0.10
car | 0.0001

This table shows that the model is selecting from many possibilities, not printing a single fact.

As a result:

- outputs vary
- hallucinations happen
- temperature matters

### LLM pipeline

The pipeline can be visualized as:

Input Text
↓
Tokenization
↓
Transformer Network
↓
Probability Distribution
↓
Token Sampling
↓
Generated Output

### Tokens section

- Models see tokens, not whole words.
- Pricing is based on tokens, not characters.
- Context windows limit how many tokens a model can consider at once.
- APIs are stateless: each request starts fresh unless you send prior context.

## Over to VS Code

Now we make the first API call and inspect the response format.

In [19]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "Teach me addition in mathematics."
        }
    ]
)

print("========== RESPONSE ==========")
print(response.choices[0].message.content)

# print("========== TOKEN USAGE ==========")
# print(f"Input Tokens  : {response.usage.prompt_tokens}")
# print(f"Output Tokens : {response.usage.completion_tokens}")
# print(f"Total Tokens  : {response.usage.total_tokens}")

========== RESPONSE ==========
Of course! Addition is one of the basic operations in mathematics, and it involves finding the total or sum by combining two or more numbers. Here’s a simple guide to understanding addition:

### Basic Concepts:

1. **Numbers**: The values you are adding together are called addends. The result of the addition is called the sum.
   - Example: In the equation \( 2 + 3 = 5 \), the numbers 2 and 3 are the addends, and 5 is the sum.

2. **Addition Symbol**: The symbol for addition is \( + \).

### How to Add:

1. **Single-Digit Addition**:
   - Start with simple numbers from 0 to 9.
   - Example: 
     - \( 1 + 1 = 2 \)
     - \( 4 + 5 = 9 \)

2. **Using Fingers**: 
   - You can also use your fingers to count. For example, to solve \( 2 + 3 \):
     - Hold up 2 fingers on one hand and 3 fingers on the other. Count all the fingers together to find the sum (2 + 3 = 5).

3. **Count On**:
   - Start with the first number and count up.
   - Example: For \( 4 + 3 \)

Expected Observation:
- You should see a short machine learning description returned by the model.
- The response includes token usage metrics.
- `prompt_tokens` is the cost of the input, `completion_tokens` is the cost of the output, and `total_tokens` is the sum.

### Request-response flow

- The `messages` array carries the full conversation history.
- APIs are synchronous: you submit the request and wait for the response.
- The model returns a structured object with text, usage, and metadata.
- This is the core pattern for most LLM engineering tasks.

# Roles section

LLM chat APIs use role-based messages to define how the AI should behave.

Role | Meaning
--- | ---
system | high-level behavior
user | user input
assistant | previous AI response

The `system` role is especially important because it can shape tone, style, and behavior.

In [ ]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a teacher for preschool kids."
        },
        {
            "role": "user",
            "content": "Teach me."
        }
    ]
)

print(response.choices[0].message.content)

Sure! Let's learn about addition. Addition means putting things together to find out how many you have in total.

For example, if you have 2 apples 🍎🍎 and you get 1 more apple 🍎, how many apples do you have now?

Let's count: 2 apples + 1 apple = 3 apples.

So, 2 + 1 = 3.

Want to try one? If you have 3 bananas 🍌🍌🍌 and get 2 more 🍌🍌, how many bananas do you have? Let's count together!


Expected Observation:
- The output should reflect a sarcastic voice if the system prompt is honored.
- This demonstrates how the `system` role influences behavior in a chat flow.

# Part 2 — Using LLM APIs for real-world tasks

LLMs are powerful text transformation engines. We will use them for summarization, extraction, and classification.

### Task 1 — Summarization

Summarization is common in enterprise applications like meeting notes, legal summaries, and support ticket digests.

In [21]:
text = """
Artificial Intelligence is transforming industries by enabling
automation, predictive analytics, and intelligent decision-making.
Companies use AI in healthcare, finance, retail, logistics, and many
other sectors.
"""

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": f"Summarize this in 1 sentence:\n\n{text}"
        }
    ]
)

print(response.choices[0].message.content)

Artificial Intelligence is revolutionizing various industries by enabling automation, predictive analytics, and intelligent decision-making across sectors such as healthcare, finance, retail, and logistics.


Expected Observation:
- The model should produce a concise single-sentence summary.
- Summarization is useful for executive summaries, support ticket triage, and document abstracts.

### Task 2 — Information extraction

Extraction turns unstructured text into structured data, which is essential for automation and analytics.

In [5]:
email = """
Hi team,

My internet has not been working since yesterday.
Please send a technician tomorrow.

Customer ID: 77821

Thanks,
Alex
"""

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": f"""
Extract:
1. customer_id
2. issue
3. requested_action

Return as JSON.

Email:
{email}
"""
        }
    ]
)

print(response.choices[0].message.content)

```json
{
  "customer_id": "77821",
  "issue": "My internet has not been working since yesterday",
  "requested_action": "Please send a technician tomorrow"
}
```


In [24]:
type(response.choices[0].message.content)

str

Expected Observation:
- The response should be valid JSON containing `customer_id`, `issue`, and `requested_action`.
- Information extraction is a practical way to build structured APIs on top of text.

### Task 3 — Classification

Classification maps text into categories, which is useful for routing tickets, labeling data, and summary workflows.

print()

In [ ]:
ticket = "My payment failed but money got deducted."

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": f"""
Classify this support ticket into one category:

Categories:
- billing
- technical
- refund
- general

Ticket:
{ticket}

Return only category name.
"""
        }
    ]
)


print(response.choices[0].message.content)

Payment Issues


Expected Observation:
- The model should return one category label, such as `billing`.
- This demonstrates zero-shot classification and simple routing automation.

# Part 3 — Prompt structuring & iteration

Good prompts make LLMs more reliable. Weak prompts are vague; strong prompts are clear, constrained, and reproducible.

### Weak prompts vs strong prompts

- Weak prompt: vague instructions, no format guidance.
- Strong prompt: explicit task, constraints, and output expectations.
- Better prompts reduce ambiguity and make outputs easier to parse.

In [7]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": "Explain neural networks"
        }
    ]
)

print(response.choices[0].message.content)

Certainly! Here's an overview of **neural networks**:

### What is a Neural Network?

A **neural network** is a type of machine learning model inspired by the structure and function of the human brain. It is designed to recognize patterns and solve complex problems by learning from data.

### Basic Structure

A neural network consists of layers of interconnected nodes called **neurons** or **units**:

1. **Input Layer**: Receives the raw data (features). Each neuron in this layer corresponds to one feature.
2. **Hidden Layers**: One or more layers where computation happens. Neurons in these layers transform inputs from the previous layer using weighted connections and activation functions.
3. **Output Layer**: Produces the final result or prediction.

### How Does It Work?

- **Weights and Biases**: Each connection between neurons has an associated weight; neurons also have biases. These parameters determine the strength and activation of signals.
- **Forward Propagation**: Data flows 

### Improved prompt

A stronger prompt provides audience, style, and format constraints.

In [8]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": """
Explain neural networks to a 10-year-old kid.

Use:
- simple language
- real-world analogies
- maximum 5 bullet points
"""
        }
    ]
)

print(response.choices[0].message.content)

Sure! Here’s a simple explanation:

- Imagine your brain is like a big network of tiny light bulbs that can turn on and off.
- Each light bulb talks to other bulbs to help you understand things, like recognizing your favorite toy.
- A neural network is like a computer’s way of copying how your brain’s light bulbs work together.
- It learns by practicing a lot, just like you get better at riding a bike with practice.
- So, neural networks help computers learn and make smart choices, like figuring out pictures or understanding words!


Expected Observation:
- The second response should be simpler, more structured, and easier to follow.
- This shows how prompt constraints improve reliability and formatting control.

### Prompt structure components

Component | Purpose
--- | ---
Role | who AI behaves as
Task | what AI should do
Context | background information
Constraints | rules
Output Format | JSON, bullets, table

Bad prompt example:

- "Explain account problems."

Good prompt example:

- "Explain in 2 sentences why a payment might fail, using simple bullet points and no technical terms."

### Few-shot prompting

Few-shot prompts give the model examples before asking it to complete a new task. This is useful when you want predictable categories or formats without training.

In [32]:
prompt = """
Classify the email into one category.

Examples:

Email: "I was charged twice this month."
Category: billing

Email: "I can't reset my password."
Category: technical

Email: "I want to cancel my subscription."
Category: account

Classify it to billing, technical, or account

Now classify this email:
[Sales, Support, Billing, technical, account]
Email: "My item is not delivered due to cancellation"

If you are not sure mention unknown category

Category:
"""

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

Category: account


Expected Observation:
- The model should follow the examples and return a category label consistent with the pattern.
- Few-shot prompting helps the model understand the expected output format.

### Chain of thought

Chain of thought prompts encourage the model to reason through steps explicitly. This can improve accuracy for harder decisions.

In [33]:
prompt = """
You are a support ticket classifier.

Analyze step-by-step:

1. Identify the issue
2. Identify secondary issues
3. Decide best category

Categories:
- billing
- technical
- refund
- account

Ticket:
"I updated my card details but payment still failed."

Give response as json in this format:
{
    "category": "billing",
    "priority": "high"
}
"""

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

{
    "category": "billing",
    "priority": "high"
}


Expected Observation:
- The model should provide step-by-step reasoning before choosing a category.
- This technique helps with interpretability and debugging.

### Structured JSON output

When you need predictable outputs, ask for strict JSON and validate it in code.

In [46]:
import json

response_text = '''
{
    "category": "billing",
    "priority": "high"
}
'''

data = json.loads(response_text)

print(data)

{'category': 'billing', 'priority': 'high'}


In [12]:
type(data)

dict

Expected Observation:
- The JSON string should parse successfully.
- In production, you should use try-except validation and allowed-value checks to avoid bad data.

## Try This Yourself

Take one of the previous examples and modify the prompt to return a table, a JSON schema, or a different category set.

# Part 4 — Temperature & randomness

This section explains how temperature impacts creativity and reliability.

### Probabilistic outputs

- Lower temperature makes outputs more deterministic.
- Higher temperature increases diversity and creativity.
- Engineers balance creativity and reliability based on use case.

In [ ]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    temperature=0.1,
    messages=[
        {
            "role": "user",
            "content": "Explain AI briefly."
        }
    ]
)

print(response.choices[0].message.content)

print("========== TOKEN USAGE ==========")
print(f"Input Tokens  : {response.usage.prompt_tokens}")
print(f"Output Tokens : {response.usage.completion_tokens}")
print(f"Total Tokens  : {response.usage.total_tokens}")

Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think, learn, and make decisions. It enables computers and systems to perform tasks that typically require human intelligence, such as understanding language, recognizing patterns, solving problems, and making predictions. AI can be categorized into narrow AI, which is designed for specific tasks, and general AI, which aims to perform any intellectual task a human can do.
========== TOKEN USAGE ==========
Input Tokens  : 11
Output Tokens : 86
Total Tokens  : 97


### High temperature

Higher temperature often makes the model more imaginative and less predictable.

In [42]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    temperature=2,
    messages=[
        {
            "role": "user",
            "content": "Explain AI briefly."
        }
    ]
)

print(response.choices[0].message.content)


print("========== TOKEN USAGE ==========")
print(f"Input Tokens  : {response.usage.prompt_tokens}")
print(f"Output Tokens : {response.usage.completion_tokens}")
print(f"Total Tokens  : {response.usage.total_tokens}")

AI, or Artificial Intelligence, refers to the simulation of human intelligence processes by machines, especially computers. It involves creating systems that can perform tasks typically requiring human intelligence, such as learning, problem-solving, understanding language, recognizing patterns, and making decisions. There are various types of AI, from narrow AI, designed for specific tasks, to advanced forms aiming for generalized human-like intelligence.
========== TOKEN USAGE ==========
Input Tokens  : 11
Output Tokens : 76
Total Tokens  : 87


Expected Observation:
- The low-temperature answer should be more direct and consistent.
- The high-temperature answer may be more expressive and varied.
- This is the creativity vs reliability tradeoff.

### Consistency testing

We can run the same prompt multiple times to measure variability.

In [43]:
results = []

prompt = "Classify this ticket as billing or technical: Payment failed error shown."

for i in range(5):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    output = response.choices[0].message.content.strip()

    results.append(output)

print(results)
print("Unique outputs:", len(set(results)))

['This ticket should be classified as **billing**.', 'This ticket should be classified as **billing**.', 'This ticket should be classified as **billing**.', 'This ticket should be classified as **billing**.', 'This ticket should be classified as **billing**.']
Unique outputs: 1


Expected Observation:
- This should show whether the same prompt gives the same label repeatedly.
- In production, normalize outputs and enforce expected values before using them.

# Part 5 — Hallucinations

Hallucinations are when the model generates plausible-sounding but incorrect information. Engineers need to avoid blind trust.

### Why hallucinations happen

- The model is predicting likely continuations, not verifying facts.
- It may invent dates, names, or citations when the prompt is ambiguous.
- Confident-sounding text can still be wrong.

### Engineering safeguards

- validation
- retries
- monitoring
- human review

Build systems that verify outputs instead of trusting raw model text.

In [44]:
import json

def parse_llm_response(response_text):
    try:
        data = json.loads(response_text)
        required_fields = [
            "category",
            "priority"
        ]

        for field in required_fields:
            if field not in data:
                raise ValueError(f"Missing field: {field}")

        return data
    except Exception as e:
        print("Validation failed:", e)
        return None

In [49]:
parse_llm_response('''
{
    "category": "billing",
    "priority": "high"
}
''')

{'category': 'billing', 'priority': 'high'}

Expected Observation:
- The function should load JSON only when the required fields are present.
- This pattern helps catch hallucinations and malformed responses early.

# Part 6 — Evaluation

Every engineering deployment needs evaluation metrics and reliability checks.

### Human evaluation and classification evaluation

- Human evaluation checks whether outputs meet quality expectations.
- Classification evaluation measures accuracy against labeled examples.
- Format validation ensures the model returns data in the expected structure.

In [ ]:
test_cases = [
    {"ticket": "I was double charged", "expected": "billing"},
    {"ticket": "I cannot login", "expected": "technical"},
]

correct = 0

for case in test_cases:

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "user",
                "content": f"""
Classify:
{case['ticket']}

Categories:
billing or technical

Return only label.
"""
            }
        ]
    )

    predicted = response.choices[0].message.content.strip().lower()

    if predicted == case['expected']:
        correct += 1

accuracy = correct / len(test_cases)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [57]:
for i in test_cases:
    print(i['ticket'])

I was double charged
I cannot login


In [51]:
80/100

0.8

Expected Observation:
- You should see a simple accuracy value between 0 and 1.
- This is a basic evaluation check for classification quality.

### Latency & cost evaluation

We measure how long a request takes and how many tokens it consumes.

In [62]:
start = time.time()

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": "Explain AI briefly in 5 words."
        }
    ]
)

latency = time.time() - start

print("Latency:", latency)

print("Input Tokens:", response.usage.prompt_tokens)
print("Output Tokens:", response.usage.completion_tokens)
print("Total Tokens:", response.usage.total_tokens)

Latency: 0.7559218406677246
Input Tokens: 15
Output Tokens: 6
Total Tokens: 21


In [63]:
response.choices[0].message.content

'Machines simulating human intelligence.'

Expected Observation:
- This prints how long the call took and the token counts.
- In production, longer prompts increase cost and latency.

# Part 7 — ML vs LLM

Compare traditional machine learning with Large Language Models in terms of use cases, data, and behavior.

### Comparison table

Aspect | ML | LLM
--- | --- | ---
Data type | structured or labeled data | unstructured text and tokens
Learning goal | predict labels or values | generate probable text continuations
Output | deterministic model predictions | probabilistic text generation
Training | often supervised, feature-based | self-supervised on text corpora
Use cases | fraud detection, forecasting, recommendations | summarization, chat, extraction

### Use case examples

- Fraud detection: traditional models use structured transaction features.
- Summarization: LLMs transform long text into concise summaries.
- Forecasting: ML models predict future values from numerical data.
- Chatbots: LLMs handle conversational and open-ended responses.
- Hybrid systems: use ML for validation and LLMs for natural language generation.

# Part 8 — Cost, latency & reliability

Finally, we cover the engineering tradeoffs when deploying LLM systems.

### Cost and latency

- Token costs scale with prompt length and response length.
- Latency includes network time and model compute time.
- More complex prompts typically cost more and take longer.
- Use shorter prompts and cached context for better efficiency.

### Reliability and retries

- LLM APIs can fail transiently due to network issues or service limits.
- Automated retries help make systems robust.
- Monitoring and alerting are essential in production.

In [19]:
import time

for attempt in range(3):

    try:

        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                    "role": "user",
                    "content": "Explain AI briefly."
                }
            ]
        )

        print(response.choices[0].message.content)

        break

    except Exception as e:
        print("Retrying...", e)
        time.sleep(2)

Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think, learn, and make decisions. It enables computers and systems to perform tasks that typically require human intelligence, such as understanding language, recognizing patterns, solving problems, and making predictions. AI can be categorized into narrow AI, which is designed for specific tasks, and general AI, which aims to perform any intellectual task a human can do.


Expected Observation:
- The loop should print the response on success.
- If an exception occurs, it retries up to three times with a pause.

# Final summary

This concludes the module with the key engineering lessons from LLM APIs.

### Key takeaways

- LLM APIs are probabilistic and token-based, not deterministic knowledge sources.
- Prompt engineering is essential for reliable, structured output.
- Temperature controls creativity vs reliability.
- Hallucinations require safeguards, validation, and human review.
- Evaluation should include accuracy, latency, and cost.
- ML and LLMs have different strengths; hybrid systems often work best.
- Engineering reliability requires monitoring, retries, and schema validation.

LLMs are powerful but probabilistic systems. Reliable AI engineering comes from evaluation, safeguards, and good system design.